In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["GH_TOKEN"] = secrets.get_secret("GITHUB_SECRET")

In [ ]:

!git clone https://github.com/Vital-ar/adversarial_physics_problems_solver.git
%cd adversarial_physics_problems_solver

In [ ]:
!pip install -r requirements/kaggle.txt
!pip install -e .

In [ ]:
from huggingface_hub import login

login(token=os.environ["HF_TOKEN"], add_to_git_credential=True)

In [ ]:
!python scripts/05_train_aligntune.py --dataset verifier_e4_curated --config configs/training/aligntune_training_v3.yaml --push-to-hub


In [ ]:
!python scripts/validate_e4_loop.py --validation-path data/splits/validation_v2.jsonl --validation-repo vital-ar/physics-solver-e4-adversarial-v3 --solver-adapter vital-ar/adversarial-physics-solver-e4-adversarial-v3-lora --verifier-adapter vital-ar/adversarial-physics-verifier-e4-curated-v3-lora --max-iterations 3 --batch-size 4 --ddp --curated-first --output-dir results/eval_e4_loop_v3_val


In [ ]:
!python scripts/validate_e4_loop.py --validation-path data/splits/test_v2.jsonl --validation-repo vital-ar/physics-solver-e4-adversarial-v3 --solver-adapter vital-ar/adversarial-physics-solver-e4-adversarial-v3-lora --verifier-adapter vital-ar/adversarial-physics-verifier-e4-curated-v3-lora --max-iterations 3 --batch-size 4 --ddp --curated-first --output-dir results/eval_e4_loop_v3_test


In [ ]:
!python scripts/07_build_experiment_tables.py


In [ ]:
%%bash
EXP_NAME='retrain-e4v-and-loop'

git config --global user.name "vital-ar"
git config --global user.email "vitalii.lvaryk.2024@gmail.com"

git checkout -B "train-v3-${EXP_NAME}"

git remote set-url origin https://${GH_TOKEN}@github.com/Vital-ar/adversarial_physics_problems_solver.git
git add models/ runs/ configs/ results/
git add .
git commit -m "kaggle :: add training v3 run outputs for ${EXP_NAME}" || true
git push -u origin "train-v3-${EXP_NAME}" --force